# Training pipeline, end-to-end — Leslie 2-Gen with Contraction

This notebook runs the latent-dynamics pipeline end-to-end for the two-generation Leslie map composed with contracting dynamics (**10D -> 2D** latent). The latent Morse graph is bistable: a period-6 orbit and an annulus.

Every stage runs in order:

> generate data -> fit scaler -> **train** the autoencoder -> diagnose -> **Conley-Morse graph (CMGDB)** -> render -> metrics

**Runtime:** ~15-25 min (the finer 27/29/30 CMGDB is the long pole); lower `cfg.training.epochs` for a quicker pass. Each run writes to its own fresh
`output/notebook_demo/<example>/run_<timestamp>/` folder, so re-running (Kernel -> Restart & Run All) never
overwrites a previous run.

In [ ]:
import os, json, time
from pathlib import Path
import latentdynamics
from latentdynamics.config import load_config
from latentdynamics.cli import pipeline

# Run from the repo root so the config's relative paths resolve.
os.chdir(Path(latentdynamics.__file__).resolve().parents[2])

cfg = load_config("configs/leslie_2gen_contraction.yaml")
# Each run writes to a fresh, timestamped folder, so re-running never overwrites
# a previous run. To reproduce again, use Kernel -> Restart & Run All.
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
run_root = Path(f"output/notebook_demo/leslie_2gen_contraction/run_{RUN_ID}")
cfg.paths.output_dir = run_root
cfg.paths.data_dir = run_root / "data"
SEED = cfg.seeds[0]
SEED_DIR = cfg.paths.output_dir / f"seed_{SEED}"
print("fresh run dir:", run_root)

# For a quicker pass, uncomment:  cfg.training.epochs = 300
print(f"system = {cfg.system.name}   dims {cfg.arch.high_dims} -> {cfg.arch.low_dims}   seed = {SEED}")
print(f"CMGDB subdiv = {cfg.cmgdb.subdiv_init}/{cfg.cmgdb.subdiv_min}/{cfg.cmgdb.subdiv_max}, padding={cfg.cmgdb.padding}")

## The pipeline

`pipeline.run(cfg, stages=[...])` runs one or more stages and persists their artifacts to disk.
We call it stage-by-stage so we can inspect each step.

## 1-2. Generate trajectory data + fit the MinMax scaler

In [ ]:
pipeline.run(cfg, stages=["data", "scale"], verbose=True)
for f in ("train.csv", "val.csv"):
    p = cfg.paths.data_dir / f
    print(f"{f}: {'ok' if p.exists() else 'MISSING'}  ({p.stat().st_size if p.exists() else 0} bytes)")

## 3. Train the autoencoder (encoder + latent map + decoder)

In [ ]:
pipeline.run(cfg, stages=["train"], verbose=True)
ts = json.load(open(SEED_DIR / "training_summary.json"))
print("epochs run:", ts.get("n_epochs_run"), " | train minutes:", round(ts.get("train_duration_minutes", 0), 2))
print("final val losses:", {k: round(v["final"], 6) for k, v in ts["val"].items()})

## 4. Diagnose the trained latent map (health checks)

In [ ]:
pipeline.run(cfg, stages=["diagnose"], verbose=True)
diag = json.load(open(SEED_DIR / "diagnose.json"))
print("diagnostic:", diag.get("diagnostic"))
print("hard_flags:", diag.get("hard_flags"))

## 5. Conley-Morse graph (CMGDB)

The expensive stage: build the box map of the trained latent dynamics, compute the Morse
decomposition and the Conley index of each Morse set. Writes `MG/morse_graph` (DOT) and
`MG/morse_sets` (CSV).

In [ ]:
pipeline.run(cfg, stages=["morse"], verbose=True)

## 6-7. Render figures + compute metrics

In [ ]:
pipeline.run(cfg, stages=["render", "metrics"], verbose=True)

## Results

The latent Morse graph (Hasse diagram), the Morse sets, and the metrics — including
`morse_graph_consistency`, which flags any attractor-type Conley index that is not a minimal
node (a sign the subdivision is too coarse).

In [ ]:
from IPython.display import Image, display
mg = SEED_DIR / "MG"
display(Image(filename=str(mg / "morse_graph.png")))
sets = mg / "morse_sets_with_overlay.png"
if not sets.exists():
    sets = mg / "morse_sets.png"
display(Image(filename=str(sets)))
m = json.load(open(SEED_DIR / "metrics.json"))
print("minimal_morse_labels:", m.get("minimal_morse_labels"))
print("consistency:", m.get("morse_graph_consistency"))